# 01 — Data Preparation: Entry Count Prediction

Implements **Steps 1–4** of the `ride-on-entry-prediction` skill.

**Target:** `entrycount` — active competitor entries per class-in-competition.

| Step | Content |
|------|---------|
| 1 | Imports + `.env` |
| 2 | Single embedded-resource query → one row per (class, prize type) |
| 3 | EDA: shape, nulls, distributions |
| 4a | Time features from `classdatetime` |
| 4b | Class dimensions: aggregates (on unique classes) + 5 classname text patterns |
| 4c | Weather via Open-Meteo archive API |
| 4d | Pivot prize rows → one row per class, presence flags, `prize_jackpot_final` (ref) |
| 4e | OHE + drop `drop_cols` + save `data/df_model.csv` |

---
## Step 1 — Imports & Environment

In [ ]:
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from dotenv import load_dotenv
from supabase import create_client

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)

load_dotenv(dotenv_path=Path("../.env"))

SUPABASE_URL = os.environ["SUPABASE_URL"]
SUPABASE_KEY = os.environ["SUPABASE_KEY"]

Path("../data/figures").mkdir(parents=True, exist_ok=True)

print(f"SUPABASE_URL : {SUPABASE_URL[:40]}...")
print("Environment  : OK")

In [2]:
import sys
!"{sys.executable}" -m pip install arabic-reshaper python-bidi


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\oren ahuvia\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


---
## Step 2 — Data Extraction

Single paginated `supabase-py` embedded-resource query.  
The reference SQL this implements (WITH subquery — one row per class per prize type):

```sql
WITH class_entries AS (
    SELECT
        cic.classincompid,
        cic.competitionid,
        cic.classdatetime,
        cic.orderinday,
        COALESCE(cic.organizercost, 0) + COALESCE(cic.federationcost, 0) AS totalcost,
        ct.classname,
        ct.classtypeid,
        f.fieldname,
        c.competitionname,
        r.ranchname,
        r.latitude,
        r.longitude,
        COUNT(CASE WHEN e.entrystatus = 'Active' THEN 1 END) AS entrycount
    FROM classincompetition cic
    JOIN classtype    ct  ON ct.classtypeid   = cic.classtypeid
    JOIN field        f   ON f.fieldid        = ct.fieldid
    JOIN competition  c   ON c.competitionid  = cic.competitionid
    JOIN ranch        r   ON r.ranchid        = c.hostranchid
    LEFT JOIN entry   e   ON e.classincompid  = cic.classincompid
    WHERE c.competitionstatus = 'הסתיימה'
    GROUP BY
        cic.classincompid, cic.competitionid, cic.classdatetime, cic.orderinday,
        cic.organizercost, cic.federationcost,
        ct.classname, ct.classtypeid, f.fieldname,
        c.competitionname, r.ranchname, r.latitude, r.longitude
)
SELECT
    ce.*,
    pt.prizetypename,
    cp.prizeamount
FROM class_entries ce
LEFT JOIN classprize cp ON cp.classincompid = ce.classincompid
LEFT JOIN prizetype  pt ON pt.prizetypeid   = cp.prizetypeid
ORDER BY ce.classdatetime, ce.classincompid, pt.prizetypename;
```

In [3]:
sb = create_client(SUPABASE_URL, SUPABASE_KEY)


def _load_raw() -> list:
    """Single paginated query via PostgREST embedded resources."""
    rows, offset = [], 0
    while True:
        resp = (
            sb.table("classincompetition")
            .select(
                "classincompid, competitionid, classdatetime, orderinday,"
                " organizercost, federationcost,"
                " classtype(classname, classtypeid, field(fieldname)),"
                " competition(competitionname, competitionstatus,"
                "   ranch(ranchname, latitude, longitude)),"
                " entry(entryid, entrystatus),"
                " classprize(prizeamount, prizetype(prizetypename))"
            )
            .range(offset, offset + 999)
            .execute()
        )
        rows.extend(resp.data)
        if len(resp.data) < 1000:
            break
        offset += 1000
    return rows


def _flatten(rows: list) -> pd.DataFrame:
    """
    Flatten nested JSON to one row per (classincompid, prizetypename).
    Mirrors the WITH-subquery SQL:
      - WHERE competitionstatus = 'הסתיימה'
      - COUNT(CASE WHEN entrystatus='Active') AS entrycount
      - LEFT JOIN classprize / prizetype (NULL row for classes with no prizes)
    """
    records = []
    for r in rows:
        comp  = r.get("competition") or {}
        if comp.get("competitionstatus") != "הסתיימה":
            continue
        ct    = r.get("classtype") or {}
        field = ct.get("field") or {}
        ranch = comp.get("ranch") or {}

        entrycount = sum(
            1 for e in (r.get("entry") or [])
            if e.get("entrystatus") == "Active"
        )
        base = {
            "classincompid":  r["classincompid"],
            "competitionid":  r["competitionid"],
            "classdatetime":  r.get("classdatetime"),
            "orderinday":     r.get("orderinday"),
            "totalcost":      float(r.get("organizercost") or 0)
                              + float(r.get("federationcost") or 0),
            "classname":      ct.get("classname"),
            "classtypeid":    ct.get("classtypeid"),
            "fieldname":      field.get("fieldname"),
            "competitionname":comp.get("competitionname"),
            "ranchname":      ranch.get("ranchname"),
            "latitude":       ranch.get("latitude"),
            "longitude":      ranch.get("longitude"),
            "entrycount":     entrycount,
        }

        prizes = r.get("classprize") or []
        if not prizes:
            # Class with no prizes → one NULL-prize row (mirrors LEFT JOIN)
            records.append({**base, "prizetypename": None, "prizeamount": None})
        else:
            for p in prizes:
                pt_name = (p.get("prizetype") or {}).get("prizetypename")
                records.append({
                    **base,
                    "prizetypename": pt_name,
                    "prizeamount":   float(p["prizeamount"])
                                     if p.get("prizeamount") is not None else None,
                })

    df = pd.DataFrame(records)
    df["classdatetime"] = pd.to_datetime(df["classdatetime"], errors="coerce")
    df["orderinday"]    = pd.to_numeric(df["orderinday"],    errors="coerce")
    df["prizeamount"]   = pd.to_numeric(df["prizeamount"],   errors="coerce")
    return df.sort_values(["classdatetime", "classincompid", "prizetypename"]).reset_index(drop=True)


raw = _load_raw()
df  = _flatten(raw)

print(f"Fetched {len(raw):,} raw rows")
print(f"After filter : {df.shape[0]:,} rows, {df['classincompid'].nunique():,} unique classes")
print(f"Columns ({df.shape[1]}): {list(df.columns)}")
df.head(6)

Fetched 1,291 raw rows
After filter : 1,049 rows, 965 unique classes
Columns (15): ['classincompid', 'competitionid', 'classdatetime', 'orderinday', 'totalcost', 'classname', 'classtypeid', 'fieldname', 'competitionname', 'ranchname', 'latitude', 'longitude', 'entrycount', 'prizetypename', 'prizeamount']


,classincompid,competitionid,classdatetime,orderinday,totalcost,classname,classtypeid,fieldname,competitionname,ranchname,latitude,longitude,entrycount,prizetypename,prizeamount
0,619,15,2024-06-06 06:00:00+00:00,1,500.000,Non Pro NRHA,12,ריינינג,תחרות בוגרים 3+4 - 2024,דאבל קיי,32.726,35.468,13,ג'קפוט,140.000
1,619,15,2024-06-06 06:00:00+00:00,1,500.000,Non Pro NRHA,12,ריינינג,תחרות בוגרים 3+4 - 2024,דאבל קיי,32.726,35.468,13,כסף מוסף,4000.000
2,620,15,2024-06-06 06:00:00+00:00,2,360.000,Limited Non Pro NRHA,13,ריינינג,תחרות בוגרים 3+4 - 2024,דאבל קיי,32.726,35.468,15,ג'קפוט,90.000
3,620,15,2024-06-06 06:00:00+00:00,2,360.000,Limited Non Pro NRHA,13,ריינינג,תחרות בוגרים 3+4 - 2024,דאבל קיי,32.726,35.468,15,כסף מוסף,1200.000
4,621,15,2024-06-06 06:00:00+00:00,3,400.000,Limited Open NRHA,8,ריינינג,תחרות בוגרים 3+4 - 2024,דאבל קיי,32.726,35.468,8,ג'קפוט,105.000
5,621,15,2024-06-06 06:00:00+00:00,3,400.000,Limited Open NRHA,8,ריינינג,תחרות בוגרים 3+4 - 2024,דאבל קיי,32.726,35.468,8,כסף מוסף,2000.000


---
## Step 3 — EDA

In [4]:
uniq = df.drop_duplicates(subset=["classincompid"])
print(f"Total rows   : {df.shape[0]:,}  (one per class×prize type)")
print(f"Unique classes: {uniq.shape[0]:,}")
print()
print("Null counts (full DataFrame):")
print(df.isnull().sum().to_string())
print()
print("Data types:")
print(df.dtypes.to_string())

Total rows   : 1,049  (one per class×prize type)
Unique classes: 965

Null counts (full DataFrame):
classincompid        0
competitionid        0
classdatetime        0
orderinday           0
totalcost            0
classname            0
classtypeid          0
fieldname            0
competitionname      0
ranchname            0
latitude            70
longitude           70
entrycount           0
prizetypename      692
prizeamount        692

Data types:
classincompid                    int64
competitionid                    int64
classdatetime      datetime64[us, UTC]
orderinday                       int64
totalcost                      float64
classname                          str
classtypeid                      int64
fieldname                          str
competitionname                    str
ranchname                          str
latitude                       float64
longitude                      float64
entrycount                       int64
prizetypename                      

In [ ]:
# entrycount distribution (one point per unique class)
ec = uniq["entrycount"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(ec, bins=30, color="steelblue", edgecolor="white", linewidth=0.5)
axes[0].set_title("Entry Count Distribution")
axes[0].set_xlabel("entries per class")
axes[0].set_ylabel("frequency")

axes[1].hist(np.log1p(ec), bins=30, color="darkorange", edgecolor="white", linewidth=0.5)
axes[1].set_title("log1p(Entry Count) Distribution")
axes[1].set_xlabel("log1p(entries)")
axes[1].set_ylabel("frequency")

plt.suptitle(f"Target variable: entrycount  (n={len(ec):,} classes)", y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig("../data/figures/entrycount_distribution.png", dpi=200, bbox_inches="tight")
plt.show()
print(ec.describe().round(2).to_string())

In [ ]:
import arabic_reshaper
from bidi.algorithm import get_display

if "fieldname" in uniq.columns and uniq["fieldname"].notna().any():
    order = (
        uniq.groupby("fieldname")["entrycount"]
        .median()
        .sort_values(ascending=False)
        .index
    )

    # Fix Hebrew labels
    fixed_order = [get_display(arabic_reshaper.reshape(str(label))) for label in order]

    fig, ax = plt.subplots(figsize=(13, 5))
    sns.boxplot(data=uniq, x="fieldname", y="entrycount", order=order, palette="Set2", ax=ax)
    ax.set_xticklabels(fixed_order, rotation=35, ha="right")
    ax.set_title("Entry Count by Field (sorted by median)")
    ax.set_xlabel("")
    plt.tight_layout()
    plt.savefig("../data/figures/entrycount_by_field.png", dpi=200, bbox_inches="tight")
    plt.show()
else:
    print("fieldname column not available.")

---
## Step 4 — Feature Engineering

In [7]:
# 4a — Time features from classdatetime
# Applied to multi-row df (classdatetime is identical for all prize rows of a class)
df_feat = df.copy()

df_feat["class_date"]  = df_feat["classdatetime"].dt.normalize()   # weather merge key
df_feat["month"]       = df_feat["classdatetime"].dt.month
df_feat["day_of_week"] = df_feat["classdatetime"].dt.dayofweek     # Mon=0

print("4a done:", ["month", "day_of_week", "class_date"])
print(df_feat["month"].value_counts().sort_index().to_string())

4a done: ['month', 'day_of_week', 'class_date']
month
3      77
4     198
5      80
6     107
8     114
9     201
10    136
11    107
12     29


In [8]:
# 4b — Class dimensions
# Aggregate features computed on unique classes to avoid prize-row inflation,
# then merged back to all rows.

_uniq = df_feat.drop_duplicates(subset=["classincompid"])

_cpc = (
    _uniq.groupby("competitionid")["classincompid"]
    .count()
    .rename("classes_per_competition")
    .reset_index()
)
_fap = (
    _uniq.groupby("fieldname")["entrycount"]
    .mean()
    .rename("field_avg_past_entries")
    .reset_index()
)
_cap = (
    _uniq.groupby("classname")["entrycount"]
    .mean()
    .rename("classname_avg_past_entries")
    .reset_index()
)

df_feat = (
    df_feat
    .merge(_cpc, on="competitionid", how="left")
    .merge(_fap, on="fieldname",     how="left")
    .merge(_cap, on="classname",     how="left")
)

# Classname structured encoding — 5 dimensions, Hebrew + English patterns
_cn = df_feat["classname"].fillna("").str.lower()

# Dimension 1 — Horse Level (one-hot, mutually exclusive)
df_feat["horse_futurity"] = _cn.str.contains("פטוריטי|futurity", na=False).astype(int)
df_feat["horse_novice"] = (
    _cn.str.contains("נוביס|novice|green horse", na=False)
    & ~_cn.str.contains("פטוריטי|futurity|דרבי|derby", na=False)
).astype(int)
df_feat["horse_derby"] = _cn.str.contains("דרבי|derby", na=False).astype(int)
df_feat["horse_none"] = (
    (df_feat["horse_futurity"] + df_feat["horse_novice"] + df_feat["horse_derby"]) == 0
).astype(int)

# Dimension 2 — Rider Age (one-hot, mutually exclusive)
df_feat["rider_youth"] = _cn.str.contains(
    r"נוער|עד\s*\d+|עד\s*גיל|\b1[0-9]\b|youth|young", na=False, regex=True
).astype(int)
df_feat["rider_adult_plus"] = (
    _cn.str.contains(r"40\+|50\+|בוגרים|prime time", na=False, regex=True)
    & ~df_feat["rider_youth"].astype(bool)
).astype(int)
df_feat["rider_all_ages"] = (
    (df_feat["rider_youth"] + df_feat["rider_adult_plus"]) == 0
).astype(int)

# Dimension 3 — Federation (one-hot, mutually exclusive)
df_feat["fed_NRHA"] = _cn.str.contains("nrha", na=False).astype(int)
df_feat["fed_NCHA"] = _cn.str.contains("ncha", na=False).astype(int)
df_feat["fed_EXCA"] = _cn.str.contains("exca", na=False).astype(int)
df_feat["fed_IEF"] = (
    (df_feat["fed_NRHA"] + df_feat["fed_NCHA"] + df_feat["fed_EXCA"]) == 0
).astype(int)

# Dimension 4 — Rider Level (multi-label, multiple flags can be 1)
df_feat["rider_open"] = _cn.str.contains(r"פתוח|לא מוגבל|unrestricted|\bopen\b", na=False, regex=True).astype(int)
df_feat["rider_non_pro"] = _cn.str.contains("נונ פרו|נונפרו|non pro|nonpro", na=False).astype(int)
df_feat["rider_yaroki"] = _cn.str.contains(r"ירוקי|ירוקי רוכב|רוכב ירוקי|רוכב חדש", na=False).astype(int)
df_feat["rider_pro"] = (
    _cn.str.contains(r"\bpro\b", na=False, regex=True)
    & ~df_feat["rider_non_pro"].astype(bool)
).astype(int)
df_feat["rider_limited"] = (
    _cn.str.contains("מוגבל|limited|limit rider", na=False)
    & ~_cn.str.contains("לא מוגבל|unrestricted", na=False)
).astype(int)

# Dimension 5 — Sub-type within field (multi-label, field-scoped)
# Cutting
df_feat["subtype_cow_horse"] = _cn.str.contains("קאו הורס", na=False).astype(int)
df_feat["subtype_premium"] = _cn.str.contains("פרימיום", na=False).astype(int)
df_feat["subtype_platinum"] = _cn.str.contains("פלאטינום", na=False).astype(int)

# All-Around
df_feat["subtype_trail"] = _cn.str.contains("טרייל", na=False).astype(int)
df_feat["subtype_horsemanship"] = _cn.str.contains("הורסמנשיפ", na=False).astype(int)
df_feat["subtype_pleasure"] = _cn.str.contains("פלז'ר|פלזר", na=False).astype(int)
df_feat["subtype_hunt_seat"] = _cn.str.contains("האנט סיט אקוויטיישן", na=False).astype(int)
df_feat["subtype_hunter_under_saddle"] = _cn.str.contains("האנטר אנדר סאדל", na=False).astype(int)
df_feat["subtype_walk_jog"] = _cn.str.contains("הליכה ג'וג|הליכה גוג", na=False).astype(int)

# Extreme Cowboy
df_feat["subtype_range_sorting"] = _cn.str.contains("ראנג' סורטינג", na=False).astype(int)
df_feat["subtype_intermediate"] = _cn.str.contains("intermediate", na=False).astype(int)
df_feat["subtype_young_gun"] = _cn.str.contains("young gun", na=False).astype(int)

# Practice flag
df_feat["is_practice"] = _cn.str.contains("אימון", na=False).astype(int)

_TEXT_COLS = [
    # Dimension 1 — Horse Level
    "horse_futurity", "horse_novice", "horse_derby", "horse_none",
    # Dimension 2 — Rider Age
    "rider_youth", "rider_adult_plus", "rider_all_ages",
    # Dimension 3 — Federation
    "fed_IEF", "fed_NRHA", "fed_NCHA", "fed_EXCA",
    # Dimension 4 — Rider Level
    "rider_open", "rider_non_pro", "rider_yaroki", "rider_pro", "rider_limited",
    # Dimension 5 — Sub-type within field
    "subtype_cow_horse", "subtype_premium", "subtype_platinum",
    "subtype_trail", "subtype_horsemanship", "subtype_pleasure",
    "subtype_hunt_seat", "subtype_hunter_under_saddle", "subtype_walk_jog",
    "subtype_range_sorting", "subtype_intermediate", "subtype_young_gun",
    # Practice flag
    "is_practice",
]
print("4b done.")
print(f"  aggregate : classes_per_competition, field_avg_past_entries,"
      f" classname_avg_past_entries, totalcost, orderinday")
print(f"  classname dims ({len(_TEXT_COLS)}): {_TEXT_COLS}")
print(f"  dim sums  :\n{df_feat.drop_duplicates('classincompid')[_TEXT_COLS].sum().to_string()}")

4b done.
  aggregate : classes_per_competition, field_avg_past_entries, classname_avg_past_entries, totalcost, orderinday
  classname dims (29): ['horse_futurity', 'horse_novice', 'horse_derby', 'horse_none', 'rider_youth', 'rider_adult_plus', 'rider_all_ages', 'fed_IEF', 'fed_NRHA', 'fed_NCHA', 'fed_EXCA', 'rider_open', 'rider_non_pro', 'rider_yaroki', 'rider_pro', 'rider_limited', 'subtype_cow_horse', 'subtype_premium', 'subtype_platinum', 'subtype_trail', 'subtype_horsemanship', 'subtype_pleasure', 'subtype_hunt_seat', 'subtype_hunter_under_saddle', 'subtype_walk_jog', 'subtype_range_sorting', 'subtype_intermediate', 'subtype_young_gun', 'is_practice']
  dim sums  :
horse_futurity                  21
horse_novice                   120
horse_derby                     17
horse_none                     807
rider_youth                    375
rider_adult_plus                84
rider_all_ages                 506
fed_IEF                        704
fed_NRHA                       191
fed_NCH

In [9]:
# 4c — Weather features removed
# temperature_2m_max and precipitation_sum are no longer retrieved:
# redundant with month dummies and cause 70-row dropna loss (missing ranch coords).
print("4c done — weather features removed from feature set.")

4c done — weather features removed from feature set.


In [10]:
# 4d — Pivot prize rows → one row per class

# Step 1: tag prize type on each row
_pn = df_feat["prizetypename"].fillna("").str.lower()
df_feat["_is_shovar"]      = _pn.str.contains("שובר|shovar|voucher",  regex=True)
df_feat["_is_jackpot"]     = _pn.str.contains("ג.קפוט|jackpot",          regex=True)
df_feat["_is_added_money"] = _pn.str.contains("כסף.מוסף|added.money",   regex=True)

# Step 2: per-class prize flags
prize_flags = (
    df_feat.groupby("classincompid")
    .agg(
        has_prize_shovar      = ("_is_shovar",      "max"),
        has_prize_jackpot     = ("_is_jackpot",     "max"),
        has_prize_added_money = ("_is_added_money", "max"),
        _any_prize            = ("prizetypename",   lambda x: x.notna().any()),
    )
    .reset_index()
)
prize_flags["no_prize"] = (~prize_flags["_any_prize"]).astype(int)
for _c in ["has_prize_shovar", "has_prize_jackpot", "has_prize_added_money"]:
    prize_flags[_c] = prize_flags[_c].astype(int)
prize_flags = prize_flags.drop(columns=["_any_prize"])

# Step 3: jackpot prize amount for prize_jackpot_final
jackpot_amt = (
    df_feat[df_feat["_is_jackpot"]]
    .groupby("classincompid")["prizeamount"]
    .max()
    .reset_index(name="prize_jackpot_posted_amount")
)

# Step 4: collapse to one row per class (drop raw prize + temp cols first)
_prize_raw = ["prizetypename", "prizeamount",
              "_is_shovar", "_is_jackpot", "_is_added_money"]
df_wide = (
    df_feat.drop(columns=_prize_raw)
    .drop_duplicates(subset=["classincompid"])
    .merge(prize_flags, on="classincompid", how="left")
    .merge(jackpot_amt, on="classincompid", how="left")
    .reset_index(drop=True)
)

# Step 5: prize_jackpot_final (reference only — dropped in 4e)
df_wide["prize_jackpot_final"] = (
    df_wide["prize_jackpot_posted_amount"].fillna(0) * df_wide["entrycount"]
)
df_feat = df_wide

# Fill prize amount NaNs with 0 — classes with no prize of that type get 0
for col in ["prize_shovar_amount", "prize_jackpot_posted_amount", "prize_added_money_amount"]:
    if col in df_feat.columns:
        df_feat[col] = df_feat[col].fillna(0)

print(f"4d done — pivoted to {df_feat.shape[0]:,} rows (one per class)")
print(f"Prize flags:\n{df_feat[['has_prize_shovar','has_prize_jackpot','has_prize_added_money','no_prize']].sum().to_string()}")
print(f"prize_jackpot_final range: {df_feat['prize_jackpot_final'].min():.0f}–{df_feat['prize_jackpot_final'].max():.0f}")

4d done — pivoted to 965 rows (one per class)
Prize flags:
has_prize_shovar          15
has_prize_jackpot        248
has_prize_added_money     94
no_prize                 692
prize_jackpot_final range: 0–19800


### 4e — Encoding decisions & multicollinearity note

Several feature groups are **mutually exclusive and exhaustive** — meaning one column in each group is always a linear combination of the others. Including all columns causes perfect multicollinearity (VIF = ∞), which makes individual coefficients uninterpretable (though predictions remain valid).

To fix this, we drop one "baseline" column per group. All remaining coefficients are then interpreted as *"compared to the baseline category."*

| Group | Columns | Baseline dropped |
|---|---|---|
| Horse level | horse_futurity, horse_novice, horse_derby, horse_none | `horse_none` (most common — 807/965 classes) |
| Rider age | rider_youth, rider_adult_plus, rider_all_ages | `rider_all_ages` (most common — 506/965 classes) |
| Federation | fed_IEF, fed_NRHA, fed_NCHA, fed_EXCA | `fed_IEF` (dominant local federation — 704/965 classes) |
| Weather | temperature_2m_max, precipitation_sum | Both dropped — redundant with month dummies, cause 70-row dropna loss |
| Field dummies | field_אולארונד, field_אקסטרים, field_קאטינג, field_ריינינג | Dropped — `field_avg_past_entries` captures the same signal more precisely |
| Prize | no_prize, has_prize_shovar, has_prize_jackpot, has_prize_added_money | `no_prize` (derived column — equals 1 only when all three prize flags are 0, carries no independent information) |

**Kept with known collinearity:** `classes_per_competition` (VIF ~14) — retained because it captures a real competitive dynamic (more classes = entries split across more options) that is independent of month patterns.

**Three features retained despite elevated VIF (all substantively justified):**

- `field_avg_past_entries` (VIF ~28) and `classes_per_competition` (VIF ~26) — these two correlate with each other because larger competitions tend to be held at fields with historically higher participation. Both are retained because they capture distinct real-world dynamics: `field_avg_past_entries` reflects discipline-level popularity, while `classes_per_competition` reflects within-competition entry splitting. Removing either would discard a meaningful predictor.

- `classname_avg_past_entries` (VIF ~10) — correlates with `field_avg_past_entries` because popular fields tend to have popular class types. Retained as the single strongest predictor in the model (coefficient ~5.9). Its collinearity is acknowledged but does not justify removing the most informative feature in the dataset.

In all three cases, the collinearity inflates coefficient uncertainty but does not invalidate predictions. The model's R² = 0.766 and RMSE = 3.09 reflect genuine predictive power on held-out data.

In [11]:
# 4e — One-hot encode month; drop weather, field dummies, and multicollinearity baselines
_month_dummies = pd.get_dummies(df_feat["month"], prefix="month", drop_first=True, dtype=int)
df_feat = pd.concat([df_feat, _month_dummies], axis=1)

_ohe = [c for c in df_feat.columns if c.startswith("month_")]
print(f"OHE added {len(_ohe)} columns: {_ohe}")

# Drop identifier/working columns + baselines + weather + field dummies
drop_cols = [
    "classincompid", "competitionid", "classtypeid",
    "classname", "competitionname", "ranchname",
    "classdatetime", "class_date",
    "latitude", "longitude",
    "month",                           # raw integer, replaced by month_* dummies
    "fieldname",                       # string column; signal captured by field_avg_past_entries
    "prizetypename", "prizeamount",   # raw prize cols (absent after 4d; listed for safety)
    "prize_jackpot_final",            # reference only
    # Multicollinearity baselines — one dropped per mutually-exclusive group
    "horse_none",                     # baseline: most common horse level (807/965)
    "rider_all_ages",                 # baseline: most common rider age (506/965)
    "fed_IEF",                        # baseline: dominant local federation (704/965)
    "no_prize",                       # derived — redundant given the three prize flags
    # Weather — redundant with month dummies, causes collinearity + 70 dropped rows
    "temperature_2m_max",
    "precipitation_sum",
    # Field dummies — redundant with field_avg_past_entries
    "field_אולארונד", "field_אקסטרים", "field_קאטינג", "field_ריינינג",
]
_drop_existing = [c for c in drop_cols if c in df_feat.columns]
df_model = df_feat.drop(columns=_drop_existing)

print(f"Dropped {len(_drop_existing)} columns.")
print(f"df_model: {df_model.shape[0]:,} rows × {df_model.shape[1]} cols")

OHE added 8 columns: ['month_4', 'month_5', 'month_6', 'month_8', 'month_9', 'month_10', 'month_11', 'month_12']
Dropped 17 columns.
df_model: 965 rows × 45 cols


---
## Stage 5 — Algorithm

In [12]:
# Stage 5 — Algorithm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
X = df_model.drop("entrycount", axis=1)
y = df_model["entrycount"]
print(f"X: {X.shape}  |  y: {y.shape}")
print("NaNs before dropna:", X.isna().sum()[X.isna().sum() > 0].to_dict())
X = X.dropna()
y = y.loc[X.index]
print(f"After dropna — X: {X.shape}")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)
print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")

# Fit scaler on X_train only — no data leakage
scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

model = LinearRegression()
model.fit(X_train_s, y_train)
print("✅ Model trained")

coef_df = (
    pd.DataFrame({"Feature": X.columns, "Coefficient": model.coef_})
    .sort_values("Coefficient", ascending=False)
    .reset_index(drop=True)
)
print(coef_df.to_string(index=False))
print("5 done.")

X: (965, 44)  |  y: (965,)
NaNs before dropna: {}
After dropna — X: (965, 44)
Train: 675  |  Test: 290
✅ Model trained
                    Feature  Coefficient
 classname_avg_past_entries        6.209
                    month_5        0.411
      has_prize_added_money        0.325
                   fed_EXCA        0.250
                 orderinday        0.213
           has_prize_shovar        0.138
               rider_yaroki        0.099
                 rider_open        0.097
                  totalcost        0.090
prize_jackpot_posted_amount        0.073
           subtype_platinum        0.053
                horse_derby        0.030
          subtype_cow_horse        0.025
       subtype_intermediate        0.019
               horse_novice        0.013
      subtype_range_sorting        0.000
           rider_adult_plus       -0.019
                  rider_pro       -0.023
                is_practice       -0.029
                day_of_week       -0.031
              rider_

---
## Stage 6 — Validation

In [ ]:
from sklearn import metrics
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ── Predict ───────────────────────────────────────────────────────────────────
y_pred = model.predict(X_test_s)

# ── Metrics ───────────────────────────────────────────────────────────────────
mae  = metrics.mean_absolute_error(y_test, y_pred)
mse  = metrics.mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = metrics.r2_score(y_test, y_pred)

print(f"MAE:  {mae:.2f}")
print(f"MSE:  {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²:   {r2:.3f}")
print("✅ Metrics computed")

# ── Residuals plot ────────────────────────────────────────────────────────────
residuals = y_test - y_pred
plt.figure(figsize=(10, 5))
sns.scatterplot(x=y_pred, y=residuals)
plt.axhline(0, color="tomato", linestyle="--")
plt.title("Residuals vs Predicted Values")
plt.xlabel("Predicted Entry Count")
plt.ylabel("Residual (Actual − Predicted)")
plt.tight_layout()
plt.savefig("../data/figures/residuals_vs_predicted.png", dpi=200, bbox_inches="tight")
plt.show()

# ── Actual vs Predicted ───────────────────────────────────────────────────────
plt.figure(figsize=(10, 5))
sns.scatterplot(x=y_test, y=y_pred)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="tomato", linestyle="--", label="Perfect prediction",
)
plt.title("Actual vs Predicted Entry Count")
plt.xlabel("Actual Entry Count")
plt.ylabel("Predicted Entry Count")
plt.legend()
plt.tight_layout()
plt.savefig("../data/figures/actual_vs_predicted.png", dpi=200, bbox_inches="tight")
plt.show()

# ── VIF — multicollinearity check ─────────────────────────────────────────────
X_vif = X.dropna()
vif_df = pd.DataFrame({
    "Feature": X_vif.columns,
    "VIF": [
        variance_inflation_factor(X_vif.values, i)
        for i in range(X_vif.shape[1])
    ],
}).sort_values("VIF", ascending=False).reset_index(drop=True)
vif_df["Flag"] = vif_df["VIF"].apply(lambda v: "⚠️ HIGH" if v > 10 else "")
print(vif_df.to_string(index=False))

print("6 done.")

# ── Save outputs ──────────────────────────────────────────────────────────────
# Save VIF table
vif_df.to_csv("../data/vif_results.csv", index=False)
print("✅ Saved vif_results.csv")

# Save model summary
summary = {
    "MAE": round(mae, 2),
    "MSE": round(mse, 2),
    "RMSE": round(rmse, 2),
    "R2": round(r2, 3),
    "train_rows": len(X_train),
    "test_rows": len(X_test),
    "n_features": X.shape[1],
}
pd.DataFrame([summary]).to_csv("../data/model_summary.csv", index=False)
print("✅ Saved model_summary.csv")

In [14]:
coef_df

,Feature,Coefficient
0,classname_avg_past_entries,6.209
1,month_5,0.411
2,has_prize_added_money,0.325
3,fed_EXCA,0.250
4,orderinday,0.213
5,has_prize_shovar,0.138
6,rider_yaroki,0.099
7,rider_open,0.097
8,totalcost,0.090
9,prize_jackpot_posted_amount,0.073


---
## Save & Summary

In [15]:
Path("../data").mkdir(parents=True, exist_ok=True)
df_model.to_csv("../data/df_model.csv", index=False)

print("Saved : ../data/df_model.csv")
print(f"Shape : {df_model.shape}")
print(f"\nColumns ({len(df_model.columns)}):")
for col in df_model.columns:
    nulls = df_model[col].isna().sum()
    print(f"  {col:<42} {str(df_model[col].dtype):<10} nulls={nulls}")

Saved : ../data/df_model.csv
Shape : (965, 45)

Columns (45):
  orderinday                                 int64      nulls=0
  totalcost                                  float64    nulls=0
  entrycount                                 int64      nulls=0
  day_of_week                                int32      nulls=0
  classes_per_competition                    int64      nulls=0
  field_avg_past_entries                     float64    nulls=0
  classname_avg_past_entries                 float64    nulls=0
  horse_futurity                             int64      nulls=0
  horse_novice                               int64      nulls=0
  horse_derby                                int64      nulls=0
  rider_youth                                int64      nulls=0
  rider_adult_plus                           int64      nulls=0
  fed_NRHA                                   int64      nulls=0
  fed_NCHA                                   int64      nulls=0
  fed_EXCA                                

In [16]:
# Stage 7 — Model export (single source of truth for the RideOn backend)
import json
import joblib
from datetime import date

MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ── Build export payload ─────────────────────────────────────────────────────
export = {
    "model_version": "v1",
    "trained_at": date.today().isoformat(),
    "features": list(X.columns),
    "coefficients": model.coef_.tolist(),
    "intercept": float(model.intercept_),
    "scaler_mean": scaler.mean_.tolist(),
    "scaler_scale": scaler.scale_.tolist(),
    "metrics": {
        "r2": float(r2),
        "rmse": float(rmse),
        "mae": float(mae),
        "n_train": len(X_train),
        "n_test": len(X_test),
    },
}

json_path = MODELS_DIR / "entry_prediction_model_v1.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(export, f, ensure_ascii=False, indent=2)
print(f"✅ Saved {json_path}")

# ── joblib backups ────────────────────────────────────────────────────────────
model_pkl_path = MODELS_DIR / "entry_prediction_model_v1.pkl"
scaler_pkl_path = MODELS_DIR / "entry_prediction_scaler_v1.pkl"
joblib.dump(model, model_pkl_path)
joblib.dump(scaler, scaler_pkl_path)
print(f"✅ Saved {model_pkl_path}")
print(f"✅ Saved {scaler_pkl_path}")

# ── Parity reference — full 965-row feature matrix + predictions ─────────────
# Freezes training-time feature values (incl. avg_past_entries aggregates) as
# the Phase 4 parity baseline. classincompid is recovered from df_feat when
# possible (row order/index is unchanged by the 4e column drop); otherwise a
# positional row_id is used.
if "df_feat" in dir() and "classincompid" in df_feat.columns:
    try:
        parity_index = pd.Index(
            df_feat.loc[X.index, "classincompid"].values, name="classincompid"
        )
    except Exception:
        parity_index = pd.RangeIndex(len(X), name="row_id")
else:
    parity_index = pd.RangeIndex(len(X), name="row_id")

y_pred_all = model.predict(scaler.transform(X))
parity_df = X.copy()
parity_df.index = parity_index
parity_df["predicted_entrycount"] = y_pred_all

parity_path = MODELS_DIR / "parity_reference_v1.csv"
parity_df.to_csv(parity_path)
print(f"✅ Saved {parity_path}  ({parity_df.shape[0]:,} rows x {parity_df.shape[1]} cols)")

# ── Self-check — reload JSON, verify lengths, and cross-check one prediction ─
with open(json_path, encoding="utf-8") as f:
    reloaded = json.load(f)

r_features = reloaded["features"]
r_coefficients = reloaded["coefficients"]
r_intercept = reloaded["intercept"]
r_mean = reloaded["scaler_mean"]
r_scale = reloaded["scaler_scale"]

assert len(r_features) == len(r_coefficients) == len(r_mean) == len(r_scale), (
    f"length mismatch: features={len(r_features)} coefficients={len(r_coefficients)} "
    f"scaler_mean={len(r_mean)} scaler_scale={len(r_scale)}"
)
print(f"✅ Length check passed — {len(r_features)} features/coefficients/scaler_mean/scaler_scale")

row = X_test.iloc[0]
manual_pred = r_intercept + sum(
    c * (row[f] - m) / s
    for f, c, m, s in zip(r_features, r_coefficients, r_mean, r_scale)
)
sklearn_pred = float(model.predict(scaler.transform(X_test.iloc[[0]]))[0])

assert abs(manual_pred - sklearn_pred) < 1e-6, (
    f"self-check mismatch: manual={manual_pred} sklearn={sklearn_pred}"
)

print(f"Manual (JSON-based) prediction : {manual_pred:.6f}")
print(f"model.predict() prediction     : {sklearn_pred:.6f}")
print("✅ Self-check passed — manual computation matches model.predict within 1e-6")
print("7 done.")

✅ Saved ..\models\entry_prediction_model_v1.json
✅ Saved ..\models\entry_prediction_model_v1.pkl
✅ Saved ..\models\entry_prediction_scaler_v1.pkl
✅ Saved ..\models\parity_reference_v1.csv  (965 rows x 45 cols)
✅ Length check passed — 44 features/coefficients/scaler_mean/scaler_scale
Manual (JSON-based) prediction : 5.052478
model.predict() prediction     : 5.052478
✅ Self-check passed — manual computation matches model.predict within 1e-6
7 done.
